# Youcat proof-of-life

This notebook exercises all of the user-table functionality provided by the youcat server (as of 2026-07-08).

Prerequisites:

* A youcat instance exists at https://data-dev.lsst.cloud/api/youcat.
* You are `danfuchs` or you have one of his tokens :)
  Right now, access to create and destroy user schemas is controlled by specifying a single OAuth subject that has that power. The youcat instance is configured to use `danfuchs`. If you need to work with this notebook and youcat instance, you can change the configuration in Phalanx.
* Nobody else is running this notebook at the same time you are. The schema and table names are hardcoded to prevent garbage from accumulating in the database from partial runs of this notebook.


Here are the imports we need. We don't use `lsst.rsp.get_tap_service` because:

* This youcat service isn't in service discovery yet
* We need to set up the auth differently than we do for our other TAP servers

In [ ]:
from lsst.rsp.utils import get_pyvo_auth
import pyvo
import requests
from io import StringIO
from requests import HTTPError

The user table functionality is behind a feature flag. We need to enable it.

In [ ]:
pyvo.utils.activate_features('cadc-tb-upload')

Create a `TAPService` instance. We need to authenticate using our auth token to these URLs in order to use the user-table functionality.

In [ ]:
url = "https://data-dev.lsst.cloud/api/youcat"

auth = get_pyvo_auth()
for path in ("sync", "async", "tables", "table-update", "load", "tmp"):
    auth.add_security_method_for_url(f"{url}/{path}", 'lsst-token', exact=False)
tap_service = pyvo.dal.TAPService(url, session=auth)

None of the user *schema* functionality is in PyVO yet. We can create and destroy the user schemas via an HTTP API that will almost certainly change in the future. Here's a `requests` session configured for this.

In [ ]:
schema_client = auth.credentials.get('lsst-token')

Here's function to delete all of the user tables and schemas in this notebook if they already exist:

In [ ]:
def cleanup(tap_service, schema_client):
    try:
        tap_service.remove_table("test_schema.test_table")
    except HTTPError as e:
        if e.response.status_code == 404:
            print("Table does not exist")
        else:
            raise
    else:
        print ("Table did exist, now it is deleted")
        
    response = schema_client.delete(f"{url}/tables/test_schema")
    try:
        response.raise_for_status()
    except HTTPError as e:
        if e.response.status_code == 404:
            print("Schema does not exist")
        else:
            print(response)
            print(response.text)
            raise e
    else:
        print("Schema did exist, now it is deleted")

Run this function to ensure we're starting fresh:

In [ ]:
cleanup(tap_service, schema_client)

Now create our user schema. This is an actual Postgres schema where our user tables will live. Again, only `danfuchs` has permission to create this schema. `danfuchs` is creating a schema that will be owned by `danfuchs`, but `danfuchs` could create a schema owned by anyone else, and then that user would be able to their own tables in that schema.

In [ ]:
schema_definition = '''<?xml version="1.0" encoding="UTF-8"?>
<vosi:tableset xmlns:vosi="http://www.ivoa.net/xml/VOSITables/v1.0" xmlns:vs="http://www.ivoa.net/xml/VODataService/v1.1" xmlns:vte="http://www.opencadc.org/xml/VOSITables-ext/v0.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <schema>
    <name>test_schema</name>
    <description>Testing user schemas</description>
  </schema>
</vosi:tableset>'''

response = schema_client.put(
    f"{url}/tables/test_schema",
    headers={
        "Content-Type": "application/x-vosi-schema",
        "x-schema-owner": "openid https://data-dev.lsst.cloud danfuchs",
    },
    data=schema_definition,
)
response.raise_for_status()

We can now start using PyVO. Most of this code is from the [PyVO docs](https://pyvo.readthedocs.io/en/latest/dal/index.html#pyvo-tap).

Let's create a table.

In [ ]:
table_definition = '''<vosi:table xmlns:vosi="http://www.ivoa.net/xml/VOSITables/v1.0" xmlns:vod="http://www.ivoa.net/xml/VODataService/v1.1" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" type="output">
    <name>my_table</name>
    <description>This is my very own table</description>
    <column>
        <name>article</name>
        <description>some thing</description>
        <dataType xsi:type="vod:VOTableType" arraysize="30*">char</dataType>
    </column>
    <column>
        <name>count</name>
        <description>how many</description>
        <dataType xsi:type="vod:VOTableType">long</dataType>
    </column>
</vosi:table>'''

tap_service.create_table(name='test_schema.test_table', definition=StringIO(table_definition))

And load it with some data:

In [ ]:
tap_service.load_table(
    name='test_schema.test_table',
    source=StringIO('article,count_of\narticle1,10\narticle2,20\n'),
    format='csv',
)

We can now query that data out of our user table!

In [ ]:
query = "SELECT * FROM test_schema.test_table WHERE article = 'article1'"
result = tap_service.run_async(query)
print(result)

We can even add indicies on our table (though we can't delete them):

In [ ]:
tap_service.create_index(table_name='test_schema.test_table', column_name='article', unique=True)

Now let's clean up by deleting our table and schema:

In [ ]:
cleanup(tap_service, schema_client)